# 00 — Start here: make one prediction understandable

**Plain-language question:** Can my Mac run the course, and what will the model
eventually do?

**Why this matters:** before training anything, you should know how to run a
notebook safely and what question a classification model answers.

**Estimated time:** 25–35 minutes.
**Prerequisite:** you can assign a Python variable and you launched Jupyter with
`make notebook`. No ML or MLflow knowledge is assumed.

This course uses fictional subscription accounts. Eventually, the model will
estimate the chance that an account cancels in the 30 days after a monthly
snapshot. Nothing here contacts Databricks or acts on a real customer.


## How a notebook works

A notebook contains two kinds of cells:

- A **Markdown cell** is explanatory text like this.
- A **code cell** runs Python and shows its result underneath.

Click the next code cell and press **Shift+Enter**. The number at its left tells
you when it ran. `[*]` means it is still running. A **kernel** is the Python
process that remembers variables between cells.

Run cells from top to bottom. If results ever seem impossible, choose
**Kernel → Restart Kernel and Run All Cells**. If setup fails, return to a
Terminal in `examples/local-classification` and run `make doctor`.

### Words introduced

| Word | Plain meaning | Example here |
|---|---|---|
| notebook | A document mixing explanation, Python, and results | This lesson |
| kernel | The Python process executing cells | `AAI Local Classification` |
| classification | Choosing between named outcomes | churn `1` or no churn `0` |


## Preflight

**Before you run this:** predict which three things a useful environment check
should report. Then run the cell.


In [ ]:
import importlib.util
import sys

required = ("mlflow", "pandas", "sklearn", "aai_local_classification")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "This notebook is using the wrong Python kernel. Close this Jupyter "
        "server, run `make notebook` from examples/local-classification, or "
        "select the 'AAI Local Classification' kernel. Missing: " + ", ".join(missing)
    )

import pandas as pd

from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings
from aai_local_classification.tracking import local_paths

settings = load_settings()
root = study_root()
paths = local_paths(root)
print(f"✓ Python {sys.version_info.major}.{sys.version_info.minor}: {sys.executable}")
print("✓ Course imports are available")
print(f"✓ Learner state: {root}")


### What you should see

Three lines beginning with `✓`: a Python 3.11 or 3.12 executable inside this
course's `.venv`, available imports, and a learner-state directory ending in
`.aai/course-v2`.

If you see those lines, the notebook is using the supported environment. That
proves only that the tools run—not that a future model is accurate or useful.


## Meet the data before the model

The next cell creates the same small synthetic dataset every time. It then loads
only the **training** rows. Later lessons explain why validation and test rows
have different jobs.

**Before you run this:** the configured course has 18 months of training data
and 120 rows per month. Predict the row count.


In [ ]:
from aai_local_classification.contracts import SplitName
from aai_local_classification.data import load_split
from aai_local_classification.workflow import ensure_prepared

manifest = ensure_prepared(settings, root)
train = load_split(settings, SplitName.TRAIN, paths.data_root)
print(f"Training shape: {train.shape}")
print(
    "Possible labels:",
    sorted(int(value) for value in train.churned_30d.unique()),
)


### What you should see

`Training shape: (2160, 12)` and labels `[0, 1]`. There are 2,160 examples and
12 stored columns. Not all 12 columns will be model inputs.

### Words introduced

| Word | Plain meaning | Concrete example |
|---|---|---|
| row / example | One case the model can learn from | one account snapshot |
| feature | A fact available when predicting | monthly fee |
| label / target | The answer learned later | `churned_30d` |


Now look at three rows. The columns on the left describe what was known at the
snapshot. The final column is the later answer.


In [ ]:
visible_columns = [
    "account_id",
    "snapshot_date",
    "monthly_fee",
    "contract_type",
    "autopay",
    "churned_30d",
]
train.loc[:2, visible_columns]


### How to read the output

Each row asks: “Using facts known on `snapshot_date`, did this fictional account
churn during the next 30 days?” A model learns a pattern across many rows; it
does not memorize a rule from the three rows shown here.


## From a probability to a yes/no prediction

A classifier can produce a score between 0 and 1. We choose a **threshold** to
turn that score into an action. This tiny example is not a trained model; it
only makes the final operation visible.

**Before you run this:** with a threshold of `0.40`, predict which scores become
`1`.


In [ ]:
toy = pd.DataFrame({"churn_probability": [0.08, 0.41, 0.76]})
toy_threshold = 0.40
toy["prediction"] = (toy.churn_probability >= toy_threshold).astype(int)
toy


### What you should see

`0.08` becomes `0`; `0.41` and `0.76` become `1`. A probability and a binary
prediction are different objects. Lesson 06 will choose a threshold using
validation data and explicit error costs.

### Misconception check

“The notebook ran” is an environment claim. It is not evidence that the data is
appropriate, the model beats a baseline, or the model should be released.


## The course roadmap

Each lesson answers one question:

1. What decision are we supporting?
2. Is the data trustworthy?
3. How do we avoid learning from the future?
4. What does “better” mean for an uncommon event?
5. How does a real training pipeline work?
6. Which model and threshold should we choose?
7. Did the fixed choice pass an honest final check?
8. What exact artifact gets released?
9. How do we notice changed behavior, and what maps to Databricks?

MLflow appears only after you understand the thing it records.


### Guided exercise

Find the minimum and maximum monthly fee observed in training. The starter cell
already selects the column; replace the two method calls if you want to explore.


In [ ]:
exercise_fee_range = train["monthly_fee"].agg(["min", "max"])
exercise_fee_range


**Self-check:** the minimum must be smaller than the maximum, and both must be
positive. This describes the observed synthetic training rows; it does not set
valid limits for future data.

<details><summary>Solution explanation</summary>

`Series.agg(["min", "max"])` applies both summaries to one column. A production
quality rule would come from a reviewed data contract, not merely these extrema.
</details>


In [ ]:
# Reference solution — run after your attempt
assert 0 < exercise_fee_range["min"] < exercise_fee_range["max"]
print("✓ Fee range is ordered and positive")


## Recap

- A kernel runs code and remembers state; top-to-bottom execution matters.
- Classification predicts a named outcome; here the label is churn `1` or no
  churn `0`.
- A score needs a threshold before it becomes a binary action.

**Evidence created:** deterministic CSV files and a manifest under the printed
learner-state directory. Rerunning this lesson reuses them after checking that
their fingerprints still match the course.

**Ready for 01?** You can explain the difference between a feature, a label, a
probability, and a prediction.
